# Operating company LBO

A five-year services buyout: revenue/opex growth, DSO/DPO/DIO working capital, %-of-revenue capex, a term loan (IO then amortizing with a balloon), cash taxes, and an exit on trailing-twelve EBITDA.

This notebook uses the benchmark model that CFDL validates against an independent reference to the penny (see `benchmarks/`).

In [ ]:
from pathlib import Path
import cfdl_sdk

# Resolve the repo root so the notebook runs from anywhere in a checkout.
ROOT = Path.cwd()
while not (ROOT / "Cargo.toml").exists():
    ROOT = ROOT.parent
PACKS = ROOT / "packs"

## Compile

Compile the model directory to IR.

In [ ]:
model_dir = ROOT / "benchmarks/opco/lbo_buyout"
model = cfdl_sdk.compile(model_dir, packs_dir=PACKS)
print("streams:", len(model.ir["streams"]))

## Run

Run with the benchmark's configuration and apply the `opco` pack's domain metrics.

In [ ]:
results = model.run(
    config=str(model_dir / "run.json"),
    pack="opco",
)
print("status:", results.status, "| warnings:", len(results.warnings))

## Cash flows

The engine returns per-period signed cash flows; `cashflows()` gives a wide DataFrame indexed by period.

In [ ]:
cf = results.cashflows()
print('shape:', cf.shape)
cf.head()

In [ ]:
# Requires the [viz] extra (pip install cfdl-sdk[viz]).
results.plot.cumulative()

## Metrics

Core metrics (NPV/IRR/MOIC/...) plus the pack's domain metrics, with their source labelled.

In [ ]:
results.metrics_frame()

## What-if

Report the free-cash-flow-to-debt-service coverage and MOIC.

In [ ]:
m = results.metrics()
print("FCF / debt service:", round(m["domain.opco.fcf_to_debt_service"], 3))
print("MOIC:", round(m["model.moic"], 3))